# 03 - Complaint Analysis + Probabilistic Data Fusion

Adds sentiment polarity, severity score, and LDA topic distribution to each CFPB complaint, aggregates them by `(product_bucket, region)`, and joins back to the bank-churn rows.

In [ ]:
import sys, pathlib, pandas as pd
sys.path.insert(0, str(pathlib.Path('.').resolve().parent))
from code import config, data_loader, preprocess, complaint_features, fusion

cfpb = pd.read_parquet(config.PROCESSED_DIR / 'cfpb_sample.parquet')
print('CFPB sample:', cfpb.shape)

In [ ]:
cfpb = complaint_features.enrich(cfpb)
cfpb.to_parquet(config.PROCESSED_DIR / 'cfpb_enriched.parquet', index=False)
cfpb[['sentiment', 'severity', 'dominant_topic']].describe()

In [ ]:
ftable = fusion.aggregate_complaint_features(cfpb)
ftable.to_csv(config.PROCESSED_DIR / 'fusion_table.csv', index=False)
ftable

In [ ]:
bank = preprocess.clean_bank_churn(data_loader.load_bank_churn())
merged = fusion.join_to_bank(bank, ftable)
merged.to_parquet(config.PROCESSED_DIR / 'bank_merged.parquet', index=False)
print('Merged shape:', merged.shape)
merged.head(3)